In [2]:
import pandas as pd

df = pd.read_csv('Fracture_Dataset.csv')
print(df.shape)
df.head()

(1000, 9)


,SI.NO,NAME,AGE,SEX,ASSO MEDICAL PROB,H/O INJURY/SURGERY,DRUG HISTORY,FREQUENCY,avg
0,1,PATIENT_1,29,Male,no,no,no,"115, 115, 113",114.30
1,2,PATIENT_2,36,Female,no,no,no,"95,94,96",94.00
2,3,PATIENT_3,37,Male,no,no,no,"80,82,81",81.00
3,4,PATIENT_4,37,Female,no,no,no,"100, 101, 100",100.33
4,5,PATIENT_5,38,female,no,no,no,99100100.4,99.80


## Data understanding

In [3]:
df.isnull().sum()

,0
SI.NO,0
NAME,0
AGE,0
SEX,0
ASSO MEDICAL PROB,0
H/O INJURY/SURGERY,0
DRUG HISTORY,0
FREQUENCY,0
avg,0


In [4]:
df.describe()

,SI.NO,AGE,avg
count,1000.000000,1000.000000,1000.000000
mean,500.500000,56.520000,78.346700
std,288.819436,12.390445,12.431321
min,1.000000,29.000000,61.330000
25%,250.750000,45.000000,69.000000
50%,500.500000,56.500000,76.416500
75%,750.250000,67.000000,88.330000
max,1000.000000,85.000000,114.300000


## Basic Preprocessing and Cleaning

In [5]:
# df['SEX'].value_counts()
# df['ASSO MEDICAL PROB'].value_counts()
# df['H/O INJURY/SURGERY'].value_counts()
# df['DRUG HISTORY'].value_counts()

In [ ]:

df['SEX'].replace({'female': 'Female', 'male': 'Male'}, inplace=True)


df['ASSO MEDICAL PROB'].replace({'No':'no',
                                   'yes (diabetes)':'yes(diabetes)',
                                   'Yes(Diabetes,bp)':'yes(diabetes,bp)',
                                  'Yes(Diabetes,Blockage in Heart)':'yes (diabetes,heart blockage)',
                                  'yes(bp dabetes)':'yes(diabetes,bp)'}, inplace=True)

## Get Target Feature from existing feature

In [7]:
def get_target_feature(i):
    if (i<60):
        return 'Normal'
    elif (i>=60 and i<=100):
        return 'Osteopenia'
    else:
        return 'Osteoporotic'


df['output'] = df['avg'].astype(int).apply(get_target_feature)
df.head()

,SI.NO,NAME,AGE,SEX,ASSO MEDICAL PROB,H/O INJURY/SURGERY,DRUG HISTORY,FREQUENCY,avg,output
0,1,PATIENT_1,29,Male,no,no,no,"115, 115, 113",114.30,Osteoporotic
1,2,PATIENT_2,36,Female,no,no,no,"95,94,96",94.00,Osteopenia
2,3,PATIENT_3,37,Male,no,no,no,"80,82,81",81.00,Osteopenia
3,4,PATIENT_4,37,Female,no,no,no,"100, 101, 100",100.33,Osteopenia
4,5,PATIENT_5,38,Female,no,no,no,99100100.4,99.80,Osteopenia


## Divide dataset in training and testing set

In [8]:
x = df.drop(['output', 'SI.NO', 'NAME', 'FREQUENCY', 'avg'], axis=1)
y = df['output']

In [9]:
x.head()

,AGE,SEX,ASSO MEDICAL PROB,H/O INJURY/SURGERY,DRUG HISTORY
0,29,Male,no,no,no
1,36,Female,no,no,no
2,37,Male,no,no,no
3,37,Female,no,no,no
4,38,Female,no,no,no


In [10]:
x.SEX = x.SEX.map( {'Male' : 1, 'Female' : 0 } )

x['ASSO MEDICAL PROB'] = x['ASSO MEDICAL PROB'].map( {'no' : 0,
                                               'yes(diabetes)' : 1,
                                               'yes(diabetes,bp)' : 2,
                                               'yes(bp)' : 3,
                                               'yes (diabetes,heart blockage)' : 4,
                                               'kidney stone' : 5,
                                               'yes(increase in heart rate)' : 6,
                                               'yes(diabetes,kidney stone)' : 7,
                                               } )

x['H/O INJURY/SURGERY'] = x['H/O INJURY/SURGERY'].map( {'no' : 0,
                                               'vericose vein surgery' : 1,
                                               'uteres removal' : 2,
                                               'kidney stone opreration' : 3,
                                               'uterus surgery' : 4,
                                               'yes(diverticulities)' : 5,
                                               'shouler surgery' : 6,
                                               'knee surgery' : 7,
                                               'yes(open heart surgery)' : 8,
                                               } )

x['DRUG HISTORY'] = x['DRUG HISTORY'].map( {'no' : 0,
                                               'yes' : 1,
                                               'yes(ecosprin)' : 2,
                                               } )


In [11]:
x.head()

,AGE,SEX,ASSO MEDICAL PROB,H/O INJURY/SURGERY,DRUG HISTORY
0,29,1,0,0,0
1,36,0,0,0,0
2,37,1,0,0,0
3,37,0,0,0,0
4,38,0,0,0,0


In [12]:
y.value_counts()

,count
output,
Osteopenia,980
Osteoporotic,20


## Model Fitting, Training and Prediction

In [13]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier()
model.fit(x, y)

DecisionTreeClassifier()

In [14]:
prediction = model.predict([[35,1,0,2,1]])

print("Predicted Condition:", prediction[0])

Predicted Condition: Osteopenia


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


In [15]:
# from sklearn.preprocessing import StandardScaler

# sc_x = StandardScaler()
# x_train = sc_x.fit_transform(x_train)
# x_test = sc_x.transform(x_test)

# print (x_train[0:10, :])

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)
model = DecisionTreeClassifier()
model.fit(x_train, y_train)

predictions = model.predict(x_test)

score = accuracy_score(y_test, predictions)
score

1.0

In [17]:
from sklearn.linear_model import LogisticRegression


x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)
model2 = LogisticRegression()
model2.fit(x_train, y_train)

predictions = model2.predict(x_test)

score = accuracy_score(y_test, predictions)
score

1.0

## Model Dumping and Loading

In [19]:
import pickle

pickle.dump(model, open('Fracture_Detection.pkl','wb'))

In [ ]:
load_model = pickle.load(open('Fracture_Detection.pkl','rb'))

prediction = load_model.predict([ [35,1,0,2,1] ])
print("Predicted Condition:", prediction[0])

In [ ]:
y_pred_loaded = load_model.predict(x_test)

score_loaded = accuracy_score(y_test, y_pred_loaded)
score_loaded